# Linear Regression

Estimation of the average treatment effect (ATE) of receiving a yellow card in the treatment window (period 1, minutes 15–45) on three defensive-action outcomes in the post window (period 2, minutes 45–60): pressures, tackles, and fouls committed. **Caveat.** This OLS estimate is biased — yellow cards are not randomly assigned, and selection on position, match context, and unobserved player traits will contaminate the coefficient on treat_yellow_card. Adjusting for pre-treatment covariates helps, but more rigorous causal methods (Double Machine Learning) are needed to get closer to the true ATE. See notebook 04.

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [3]:
def fit_ols(formula: str, data: pd.DataFrame, cluster_col: str = "match_id"):
    """Fit OLS with cluster-robust standard errors at the match level."""
    return smf.ols(formula, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data[cluster_col]},
    )

## Data

In [4]:
df = pd.read_csv("data/analysis_frame.csv", parse_dates=["match_date"])
df["competition_format"] = (df["competition_type"] == "Domestic League").map({True: "league", False: "cup"})

print(f"Sample: {len(df):,} player-matches")
print(f"Treated: {df['treat_yellow_card'].sum():,}  ({df['treat_yellow_card'].mean():.2%})")

Sample: 80,226 player-matches
Treated: 3,210  (4.00%)


In [5]:
TREATMENT = "treat_yellow_card"

def _cols(prefix): return sorted(c for c in df.columns if c.startswith(prefix))
PRE_PLAYER_TYPE_COLS = _cols("pre_player_n_")
PRE_TEAM_TYPE_COLS   = _cols("pre_team_n_")
PRE_OPP_TYPE_COLS    = _cols("pre_opp_n_")
# str_* dropped — not defined for tournament matches; competition_format absorbs context.
PRE_ALL_COLS = (PRE_PLAYER_TYPE_COLS + PRE_TEAM_TYPE_COLS + PRE_OPP_TYPE_COLS
                + ["pre_score_diff"])
print(f"Pre-treatment numeric controls: {len(PRE_ALL_COLS)} "
      f"+ position_group + home_away + gender + competition")


Pre-treatment numeric controls: 87 + position_group + home_away + gender + competition


## OLS — naive and fully-adjusted, for each DV

Two specs per DV: (1) treatment only (the raw gap), (2) + all pre-treatment
covariates + position/home/gender/competition. Cluster-robust SE at match.


In [6]:
DVS = ["post_n_pressure", "post_n_tackle", "post_n_foul_committed", "post_n_def_events"]
controls = " + ".join(PRE_ALL_COLS)
rows = []
for dv in DVS:
    m1 = fit_ols(f"{dv} ~ {TREATMENT}", df)
    m2 = fit_ols(f"{dv} ~ {TREATMENT} + position_group + home_away + gender "
                 f"+ C(competition_format) + {controls}", df)
    base = df.loc[df[TREATMENT]==0, dv].mean()
    rows.append({"DV": dv, "control_mean": round(base,3),
                 "naive_ATE": round(m1.params[TREATMENT],4), "naive_p": round(m1.pvalues[TREATMENT],4),
                 "adj_ATE": round(m2.params[TREATMENT],4), "adj_SE": round(m2.bse[TREATMENT],4),
                 "adj_p": round(m2.pvalues[TREATMENT],4), "adj_rel_%": round(100*m2.params[TREATMENT]/base,1)})
print(pd.DataFrame(rows).to_string(index=False))


                   DV  control_mean  naive_ATE  naive_p  adj_ATE  adj_SE  adj_p  adj_rel_%
      post_n_pressure         2.145     0.6680      0.0   0.0257  0.0377 0.4957        1.2
        post_n_tackle         0.257     0.1024      0.0   0.0117  0.0113 0.2989        4.5
post_n_foul_committed         0.164     0.0381      0.0  -0.0165  0.0085 0.0512      -10.1
    post_n_def_events         2.567     0.8085      0.0   0.0208  0.0446 0.6402        0.8
